# RFM Customer Segmentation
**RetailPulse | Business Analytics**

RFM (Recency, Frequency, Monetary) analysis segments customers into actionable groups for targeted marketing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import psycopg2

conn = psycopg2.connect(
    host='postgres', dbname='retailpulse',
    user='retailpulse', password='retailpulse123'
)

In [ ]:
# Load RFM scores from Gold layer
rfm = pd.read_sql("""
    SELECT r.user_id, r.recency_days, r.frequency, r.monetary,
           r.r_score, r.f_score, r.m_score, r.rfm_total, r.rfm_segment,
           c.tier
    FROM gold.agg_rfm_scores r
    JOIN gold.dim_customers c ON r.user_id = c.user_id
""", conn)

print(f'Total customers: {len(rfm):,}')
rfm.head()

In [ ]:
# Segment distribution
seg_counts = rfm['rfm_segment'].value_counts()
seg_revenue = rfm.groupby('rfm_segment')['monetary'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = plt.cm.Set3(np.linspace(0, 1, len(seg_counts)))
axes[0].pie(seg_counts.values, labels=seg_counts.index, autopct='%1.1f%%', colors=colors)
axes[0].set_title('Customer Count by RFM Segment', fontweight='bold')

seg_revenue.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Revenue by RFM Segment', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Total Revenue (USD)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('rfm_segments.png', dpi=150)
plt.show()

In [ ]:
# RFM scatter: Recency vs Frequency, coloured by Monetary
plt.figure(figsize=(12, 7))
scatter = plt.scatter(
    rfm['recency_days'], rfm['frequency'],
    c=rfm['monetary'], cmap='YlOrRd',
    s=30, alpha=0.6, edgecolors='none'
)
plt.colorbar(scatter, label='Monetary Value (USD)')
plt.xlabel('Recency (days since last purchase)')
plt.ylabel('Frequency (number of orders)')
plt.title('RFM Customer Map — Recency vs Frequency vs Monetary', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# K-Means clustering (alternative to rule-based RFM)
features = rfm[['recency_days', 'frequency', 'monetary']].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# Elbow method
inertias = []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, marker='o', color='steelblue')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k', fontweight='bold')
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

# Fit with k=5
km = KMeans(n_clusters=5, random_state=42, n_init=10)
rfm['kmeans_cluster'] = km.fit_predict(X_scaled)
print(rfm.groupby('kmeans_cluster')[['recency_days','frequency','monetary']].mean().round(2))

## Marketing Actions by Segment

| Segment | Action |
|---|---|
| Champions | Reward, ask for reviews, upsell premium |
| Loyal Customers | Loyalty program, early access |
| At Risk | Win-back campaign, discount |
| Lost | Re-engagement email, deep discount |
| Recent Customers | Onboarding email, cross-sell |